# Agent v5 -- Interactive Test Notebook

Tests the v5 agent end-to-end against **real data** in the `enriched_property_listing` MongoDB collection (schema_version 1): tools, streaming, multi-turn memory, and structured output.

**Prerequisites:** `.env` with `AI_GATEWAY_API_KEY`, `MONGODB_PW` (v5 does not use Upstash)

In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Root:', ROOT)

Root: C:\Users\Yang Hui\Desktop\projects\landy_ai


In [2]:
import os
from dotenv import load_dotenv

load_dotenv(ROOT / '.env')

for var in ['AI_GATEWAY_API_KEY', 'MONGODB_PW']:
    status = 'OK' if os.getenv(var) else 'MISSING'
    print(f'{var}: {status}')

AI_GATEWAY_API_KEY: OK
MONGODB_PW: OK


## 1. Real data sanity check -- `enriched_property_listing`

In [3]:
from utility.property_listing_init import get_enriched_property_listing_collections

col = get_enriched_property_listing_collections()
total = col.count_documents({})
active = col.count_documents({'listing_status': 'active'})
print(f'documents: {total} total / {active} active')
print('schema_version values:', col.distinct('schema_version'))
print('offer types:', col.distinct('offer.offer_type'))
print('categories:', col.distinct('main_category'))

documents: 137 total / 137 active
schema_version values: [1]
offer types: ['rent', 'sale']


categories: ['agricultural-land', 'cluster-factory', 'detached-factory', 'factory', 'industrial-land', 'semi-d-factory', 'shoplot', 'showroom', 'terrace-factory', 'warehouse']


In [4]:
from agent.v5.tools._utils import serialize_listing

doc = col.find_one({'listing_status': 'active', 'traits.industrial': {'$exists': True}})
s = serialize_listing(doc)
for k, v in s.items():
    print(f'{k:24} {v}')

property_id              13
title                    Warehouse for Rent in Balakong – 55,713 sqft Built-Up
slug                     warehouse-for-rent-balakong-55713-sqft
thumbnail                https://pub-5cf4bc1a03ad43d0a020752835ca6de0.r2.dev/uploads/b531ea6a3489465295c27243cf5e9ea3.webp
offer_type               rent
price                    165000
currency                 MYR
price_per_sqft           2.96
city                     Balakong
state                    Selangor
industrial_park          Kawasan Perindustrian balakong Jaya
street                   
main_category            factory
sub_categories           ['detached-factory', 'distribution-center', 'factory', 'logistics-hub', 'vacant-possession', 'warehouse']
tenure                   freehold
built_up_sqft            55713
land_sqft                103455
ceiling_height_m         12.19
floor_loading_kn_m2      29.42
nearest_highway          {'name': 'Maju Expressway (MEX, E20)', 'distance_km': 6.42}
listed_date           

## 2. Direct tool test -- `find_listings` (real MongoDB query)

Outside a LangGraph run there is no stream writer, so we capture SSE events by patching `get_stream_writer` in the tool module.

In [5]:
import contextlib

@contextlib.contextmanager
def capture_events(*modules):
    """Temporarily replace get_stream_writer in tool modules with an event collector."""
    events = []
    originals = {m: m.get_stream_writer for m in modules}
    for m in modules:
        m.get_stream_writer = lambda: events.append
    try:
        yield events
    finally:
        for m, orig in originals.items():
            m.get_stream_writer = orig

In [6]:
import agent.v5.tools.find_listings as fl

with capture_events(fl) as events:
    result = await fl.find_listings.ainvoke({
        'offer_type': 'rent',
        'property_category': ['factory', 'warehouse'],
        'region': 'Selangor',
    })

print(f"total_found: {result['total_found']}")
print(f"filters_applied: {result['filters_applied']}")
print(f"location_breakdown: {result['location_breakdown']}")
print()
for r in result['property_listing_result'][:5]:
    print(f"  [{r['property_id']}] {r['title']}")
    print(f"      {r['offer_type']} | RM{r['price']:,} | {r['city']} | features: {r['extracted_key_features'][:2]}")
print()
print('SSE events:', [e['event'] for e in events])

c:\Users\Yang Hui\Desktop\projects\landy_ai\.venv\Lib\site-packages\langgraph\checkpoint\base\__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


total_found: 29
filters_applied: category=factory,warehouse, region=Selangor
location_breakdown: ['Balakong', 'Cheras', 'Petaling Jaya', 'Semenyih', 'Telok Panglima Garang', 'Subang Jaya', 'Shah Alam']

  [13] Warehouse for Rent in Balakong – 55,713 sqft Built-Up
      rent | RM165,000 | Balakong | features: ['Warehouse / industrial factory space', 'For rent in Balakong']
  [14] Terrace Factory for Rent in Esteem Business Park, Klang
      rent | RM17,504 | None | features: ['Built-up from 8,141 sqft', '30 ft ceiling height at production area (10 ft at Level 2 warehouse area)']
  [25] 2-Storey Terrace Factory for Rent in Cheras Jaya, Selangor
      rent | RM13,000 | Cheras | features: ['2-storey terrace factory (vacant)', 'Built-up: 9,600 sqft']
  [38] Terrace Factory for Rent in Selesa Jaya, Balakong, Selangor
      rent | RM11,000 | Cheras | features: ['1.5-storey link (terrace) factory', 'Land area: 4,000 sqft (40’ x 100’)']
  [28] Detached Factory for Rent in Hicom-Glenmarie, Petal

In [7]:
# Proximity filter on real data: factories within 5km of a highway
with capture_events(fl) as events:
    near_highway = await fl.find_listings.ainvoke({
        'property_category': ['factory'],
        'max_highway_km': 5.0,
    })

print(f"total_found: {near_highway['total_found']}")
for r in near_highway['property_listing_result'][:5]:
    hw = r['nearest_highway'] or {}
    print(f"  [{r['property_id']}] {r['title'][:60]} -- {hw.get('name')} @ {hw.get('distance_km')}km")

total_found: 20
  [10] Semi-D Factory for Sale in HICOM Industrial Park, Shah Alam -- Shah Alam Expressway (KESAS, E5) @ 4.39km
  [38] Terrace Factory for Rent in Selesa Jaya, Balakong, Selangor -- Shah Alam Expressway (KESAS, E5) @ 4.73km
  [35] Terrace Factory for Sale in Taman Perindustrian Selesa Jaya, -- Shah Alam Expressway (KESAS, E5) @ 4.73km
  [28] Detached Factory for Rent in Hicom-Glenmarie, Petaling Jaya -- Federal Highway (FT2) @ 1.55km
  [39] Terrace Factory for Sale in Taman Perindustrian Selesa Jaya, -- Shah Alam Expressway (KESAS, E5) @ 4.73km


## 3. Direct tool test -- `get_listing_detail`

In [8]:
import agent.v5.tools.get_listing_detail as gld

pid = result['property_listing_result'][0]['property_id']

with capture_events(gld) as events:
    detail = await gld.get_listing_detail.ainvoke({'property_id': pid})

print(f"Title:        {detail['title']}")
print(f"Description:  {(detail['description'] or '')[:200]}...")
print(f"Power supply: {detail['power_supply']}")
print(f"Loading bays: {detail['loading_bays']}")
print(f"Risk factors: {detail['risk_factors']}")
print(f"Similar IDs:  {detail['similar_listing_id']}")
print(f"Images:       {len(detail['images'])}")
print()
print('SSE events:', [e['event'] for e in events])

Title:        Warehouse for Rent in Balakong – 55,713 sqft Built-Up
Description:   Industrial Factory & Warehouse for Rent in Balakong
Searching for a high-spec industrial facility in Balakong? This freehold warehouse & factory offers 55,713 sqft built-up, 40ft ceiling height, and ...
Power supply: {'amps': 1000, 'phase': 3, 'voltage': None}
Loading bays: None
Risk factors: []
Similar IDs:  [59, 21, 82, 75, 68, 62, 64, 46, 123, 138]
Images:       14

SSE events: ['listing_detail_card']


## 4. Create agent (InMemorySaver -- no Postgres needed)

In [9]:
from langgraph.checkpoint.memory import InMemorySaver
from agent.v5.orchestration import create_agent

checkpointer = InMemorySaver()
agent = create_agent(checkpointer)

def unwrap(resp):
    """langgraph >= 1.1 returns GraphOutput; the result dict lives on .value"""
    return getattr(resp, 'value', resp)

def get_structured(resp):
    """structured_response is absent when the model skips the output tool call."""
    return unwrap(resp).get('structured_response')

print('Agent created:', type(agent))

Agent created: <class 'langgraph.graph.state.CompiledStateGraph'>


## 5. Single-turn invoke

In [10]:
THREAD_ID = 'v5-test-01'

response = await agent.ainvoke(
    {'messages': 'factory for rent in Selangor under RM200k per month'},
    {'configurable': {'thread_id': THREAD_ID}},
    version='v2',
)
out = unwrap(response)

print('=== final message ===')
print(out['messages'][-1].content)
print()
structured = get_structured(response)
if structured:
    print('follow_up_chips:   ', structured.follow_up_chips)
    print('live_agent_cta:    ', structured.live_agent_cta)
    print('live_agent_trigger:', structured.live_agent_trigger)
else:
    print('(no structured_response this turn)')

=== final message ===
Returning structured response: follow_up_chips=['Smaller terrace factories under RM20k', 'Larger options near Port Klang', 'Speak to our property agent'] live_agent_cta=False live_agent_trigger=None

follow_up_chips:    ['Smaller terrace factories under RM20k', 'Larger options near Port Klang', 'Speak to our property agent']
live_agent_cta:     False
live_agent_trigger: None


## 6. Multi-turn conversation (filter persistence)

In [11]:
THREAD_ID_MT = 'v5-test-multi-turn-01'

turns = [
    'warehouse for rent in Selangor',
    'only ones near a highway please',
    'which of these has the highest ceiling?',
]

for i, msg in enumerate(turns, 1):
    resp = await agent.ainvoke(
        {'messages': msg},
        {'configurable': {'thread_id': THREAD_ID_MT}},
        version='v2',
    )
    s = get_structured(resp)
    print(f'--- Turn {i}: {msg!r} ---')
    print(unwrap(resp)['messages'][-1].content[:400])
    print(f'chips: {s.follow_up_chips if s else None}')
    print()

--- Turn 1: 'warehouse for rent in Selangor' ---
Returning structured response: follow_up_chips=['Warehouse near Port Klang', 'Smaller units under RM30k', 'Speak to our property agent'] live_agent_cta=False live_agent_trigger=None
chips: ['Warehouse near Port Klang', 'Smaller units under RM30k', 'Speak to our property agent']



--- Turn 2: 'only ones near a highway please' ---
Returning structured response: follow_up_chips=['Lower price options', 'Larger warehouse units', 'Speak to our property agent'] live_agent_cta=False live_agent_trigger=None
chips: ['Lower price options', 'Larger warehouse units', 'Speak to our property agent']



--- Turn 3: 'which of these has the highest ceiling?' ---
Returning structured response: follow_up_chips=['View unit details', 'Filter by ceiling height', 'Speak to our property agent'] live_agent_cta=False live_agent_trigger=None
chips: ['View unit details', 'Filter by ceiling height', 'Speak to our property agent']



## 7. Streaming -- watch SSE events live

In [12]:
THREAD_ID_STREAM = 'v5-test-stream-02'

async for raw_event in agent.astream(
    {'messages': 'newest warehouses for sale in Selangor'},
    {'configurable': {'thread_id': THREAD_ID_STREAM}},
    stream_mode=['updates', 'messages', 'custom'],
    subgraphs=True,
    version='v2',
):
    # langgraph >= 1.1 yields dicts; older versions yield 3-tuples
    if isinstance(raw_event, dict):
        type_, ns, data = raw_event.get('type'), raw_event.get('ns'), raw_event.get('data')
    elif isinstance(raw_event, tuple) and len(raw_event) == 3:
        type_, ns, data = raw_event
    else:
        continue

    if type_ == 'custom':
        event_name = data.get('event', '?') if isinstance(data, dict) else '?'
        if event_name == 'search_start':
            print(f"\n[search_start] filters_active={data.get('filters_active')}")
        elif event_name == 'property_cards':
            print(f"[property_cards] {len(data.get('listings', []))} listings")
        elif event_name == 'search_complete':
            print(f"[search_complete] total_found={data.get('total_found')}")
        else:
            print(f'[custom] {event_name}')

    elif type_ == 'messages':
        if isinstance(data, tuple) and data:
            token = getattr(data[0], 'content', '')
            if token:
                print(token, end='', flush=True)

print('\n--- stream complete ---')


[search_start] filters_active=3


[property_cards] 29 listings
[search_complete] total_found=29
{"total_found": 29, "property_listing_result": [{"property_id": "141", "title": "Freehold Detached Factory with Expansive Corporate Offices at Temasya Glenmarie, Shah Alam", "slug": "freehold-detached-factory-corporate-office-temasya-glenmarie-shah-alam", "thumbnail": "https://pub-5cf4bc1a03ad43d0a020752835ca6de0.r2.dev/uploads/3ea1ec3e29ee4a4187aa97329055e53d.webp", "offer_type": "sale", "price": 19302000, "currency": "MYR", "price_per_sqft": 780.95, "city": "Shah Alam", "state": "Selangor", "industrial_park": "Temasya Glenmarie", "street": "", "main_category": "factory", "sub_categories": ["car-showroom", "cleanroom", "detached-factory", "factory", "logistics-hub", "pharma", "semiconductor", "showroom", "warehouse"], "tenure": "freehold", "built_up_sqft": 24716, "land_sqft": 22293, "ceiling_height_m": null, "floor_loading_kn_m2": null, "nearest_highway": null, "listed_date": "2026-05-11T16:48:54.186000", "ai_summary": "Fac

Returning structured response: follow_up_chips=['Refine by price range', 'Factories in Shah Alam', 'Speak to our property agent'] live_agent_cta=False live_agent_trigger=None


--- stream complete ---


## 8. Live agent CTA trigger test (transact_intent)

In [13]:
THREAD_ID_CTA = 'v5-test-cta-01'

resp = await agent.ainvoke(
    {'messages': 'I want to book a viewing and make an offer on a factory in Shah Alam'},
    {'configurable': {'thread_id': THREAD_ID_CTA}},
    version='v2',
)

s = get_structured(resp)
print(unwrap(resp)['messages'][-1].content[:400])
print()
if s:
    print('live_agent_cta:    ', s.live_agent_cta)
    print('live_agent_trigger:', s.live_agent_trigger)
    print('follow_up_chips:   ', s.follow_up_chips)
else:
    print('(no structured_response this turn)')

Returning structured response: follow_up_chips=['View factories in Shah Alam', 'Speak to our property agent', 'Check pricing details'] live_agent_cta=True live_agent_trigger='transact_intent'

live_agent_cta:     True
live_agent_trigger: transact_intent
follow_up_chips:    ['View factories in Shah Alam', 'Speak to our property agent', 'Check pricing details']


## 9. Postgres checkpointer (production mode)

Uncomment and set `DB_URI` to test with persistent memory across kernel restarts.

In [14]:
# DB_URI = os.getenv('DB_URI')

# from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver

# async with AsyncPostgresSaver.from_conn_string(DB_URI) as cp:
#     await cp.setup()
#     pg_agent = create_agent(cp)

#     resp = await pg_agent.ainvoke(
#         {'messages': 'warehouse for sale in Klang'},
#         {'configurable': {'thread_id': 'v5-prod-test-01'}},
#         version='v2',
#     )
#     print(unwrap(resp)['messages'][-1].content)